# Classes - Review

A __class__ is a body of code that defines the attributes and behaviors required to accurately model something you need for your program. You can model something from the real world, such as a rocket ship or a guitar string, or you can model something from a virtual world such as a rocket in a game, or a set of physical laws for a game engine.

An __attribute__ is a piece of information. In code, an attribute is just a variable that is part of a class.

A __behavior__ is an action that is defined within a class. These are made up of __methods__, which are just functions that are defined for the class.

An __object__ is a particular **instance** of a class. An object has a certain set of values for all of the attributes (variables) in the class. You can have as many objects as you want for any one class.

__Classes are new custom data types that you can develop.__ 

We used classes with the Hearts functions.  

We will also use classes with the unittest Python package 

# Unit tests

We will go back to our Hearts game to look at unit testing  

Unit test used to determine if a function is working as expected and providing the expected output.  
- the smallest possible unit of code in isolation
- known inputs against expected outputs
- checking whether change to one part of the code breaks something elsewhere that was previously working

Different from integration tests--do the functions work together?


|Type|Question|Hearts example|
|---|---|---|
|Unit|Does this function work correctly in isolation?|Does round_score() always sum to 26?|
|Integration|Do these functions work correctly together?|Does play_round() correctly update player scores using round_score() output?|
|End to end|Does the whole system do what it should?|Does a complete game produce a valid winner?|


# Setting up unit tests

## Steps:
- Figure out code to test
- Set up testing code
- Run tests 

# Set up testing code

- Easiest to test from the command line but we will do it in a notebook

- Create new Python script starting with "test_" so testing libraries can find it

- import testing module

- Create test class (if using unittest)


# Modularity

We can see another reason for modularity 
- By testing each function separately we can ensure that the code is easier to debug
- Rather than debugging a big block of code, we can make sure functions are working individually

In [1]:
import numpy as np
import time

rng = np.random.default_rng()

# --- Player Class ---
class Player:
    def __init__(self, player_id):
        self.player_id = player_id
        self.score = 0
        self.games_won = 0

    def add_score(self, points):
        self.score += points

    def reset_score(self):
        self.score = 0

    @property
    def is_winning(self):
        """Can be checked externally against other players"""
        return self.score  # lower is better in Hearts

    def __repr__(self):
        return f"Player {self.player_id} | Score: {self.score} | Wins: {self.games_won}"


# --- HeartsGame Class ---
class HeartsGame:
    def __init__(self):
        self.players = [Player(i) for i in range(4)]
        self.rounds_played = 0

    def round_score(self):
        """Simulate one round of Hearts"""
        scores = []
        remaining = 13
        for i, player in enumerate(self.players):
            if i < 3:
                score = rng.integers(0, remaining, endpoint=True)
                scores.append(score)
                remaining -= score
            else:
                scores.append(remaining)

        queen = rng.integers(0, 3, endpoint=True)
        scores[queen] += 13
        return scores

    def play_round(self):
        """Apply one round's scores to players"""
        new_scores = self.round_score()
        for player, score in zip(self.players, new_scores):
            player.add_score(score)
        self.rounds_played += 1

    def get_winners(self):
        """Return player(s) with the lowest score"""
        min_score = min(p.score for p in self.players)
        return [p for p in self.players if p.score == min_score]

    def reset(self):
        """Reset for a new game"""
        for player in self.players:
            player.reset_score()
        self.rounds_played = 0

    def play(self, gametype, endvalue):
        """Main game loop"""
        self.reset()

        if gametype == "points":
            while all(p.score < endvalue for p in self.players):
                self.play_round()

        elif gametype == "rounds":
            for _ in range(endvalue):
                self.play_round()

        winners = self.get_winners()
        for w in winners:
            w.games_won += 1

        player_list = ', '.join(str(w.player_id) for w in winners)
        label = "winner is player" if len(winners) == 1 else "winners are players"
        #print(f"Yay! The {label} {player_list} after {self.rounds_played} rounds")
        #print(', '.join(str(p.score) for p in self.players))
        #print("Standings:", self.players)

# round_score function

We saw that the `round_score` function is biased.  
Player 0 rarely wins  


In [2]:
game = HeartsGame()
i = 1
while i<5000:
    game.play("points", 100)
    i+=1
print("Standings: ")
for i, p in enumerate(game.players):
    print(f"Player {i}: "+ str(game.players[i].games_won))    

Standings: 
Player 0: 32
Player 1: 787
Player 2: 2136
Player 3: 2145


# Debugging

This is the most difficult type of error to debug: a **logical** error
- The code is working and returning output
- There is nothing obviously wrong with the output

Player 0 does win ocassionally and has a score so it's not an indexing issue  
There must be something within the function


```
    def round_score(self):
        """Simulate one round of Hearts"""
        scores = []
        remaining = 13
        for i, player in enumerate(self.players):
            if i < 3:
                score = rng.integers(0, remaining, endpoint=True)
                scores.append(score)
                remaining -= score
            else:
                scores.append(remaining)

        queen = rng.integers(0, 3, endpoint=True)
        scores[queen] += 13
        return scores
```
# Any ideas? 

# Testing expected output

We will see how to test the function using both `pytest` and `unittest`

We know what to expect: 
As we run more simulations, each player should get about a quarter of the points

Also, the tests should really be run from the command line rather than in juPyter notebooks


In [3]:
import pytest
def test_round_score_unbiased():
    """Over many rounds, each player should receive similar total hearts"""
    totals = [0] * 4
    for _ in range(10000):
        scores = game.round_score()
        for i, s in enumerate(totals):
            totals[i] += scores[i]
    # Each player should have roughly 25% of total hearts
    for total in totals:
        #self.assertAlmostEqual(total / sum(totals), 0.25, delta=0.02)
        assert total / sum(totals) == pytest.approx(0.25, abs=0.02)


# Reading the code

`assert total / sum(totals) == pytest.approx(0.25, abs=0.02)` 

We are testing whether the actual percentage of hearts received for each player is about 25%  
We could adjust `abs=0.02` if we feel the range is too narrow.

Also, we ran the simulation 10,000 times: `for _ in range(10000):`

I will move the function to a separate .py file to run the test

In [4]:
import unittest
import numpy as np
class TestHeartsGame(unittest.TestCase):
    
    def setUp(self):
        """Create a fresh game before each test"""
        self.game = HeartsGame()

    def test_round_score_unbiased(self):
        """Over many rounds, each player should receive similar total hearts"""
        totals = [0] * 4
        for _ in range(10000):
            scores = game.round_score()
            for i, s in enumerate(totals):
                totals[i] += scores[i]
        # Each player should have roughly 25% of total hearts
        for total in totals:
            self.assertAlmostEqual(total / sum(totals), 0.25, delta=0.02)

# Unittest code

The unittest code is largely the same.  
But note the two major differences:  
1) We have created a new class
2) The unittest classes have their own `assert` methods  
   We are using `unittest`'s `assertAlmostEqual` method here  

In [5]:
# Both pytest and unittest work better in the console but we can see the results here 
!pytest test_hearts_pytest.py -v

============================= test session starts =============================
platform win32 -- Python 3.11.14, pytest-9.0.2, pluggy-1.5.0 -- C:\Users\arpie\anaconda3\envs\QMSS\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\arpie\Documents\QMSS\QMSS-GR5072-Spring2026\Week 8
plugins: anyio-4.10.0
collecting ... collected 1 item

test_hearts_pytest.py::test_round_score_unbiased FAILED                  [100%]

================================== FAILURES ===================================
__________________________ test_round_score_unbiased __________________________

    def test_round_score_unbiased():
        """Over many rounds, each player should receive similar total hearts"""
        game = HeartsGame()
        totals = [0] * 4
        for _ in range(10000):
            scores = game.round_score()
            for i, s in enumerate(totals):
                totals[i] += scores[i]
        # Each player should have roughly 25% of total hearts
        for total in totals:
      

In [6]:
suite = unittest.TestLoader().loadTestsFromNames(['test_round_score_unbiased'], TestHeartsGame)
unittest.TextTestRunner(verbosity=2).run(suite)

test_round_score_unbiased (__main__.TestHeartsGame.test_round_score_unbiased)
Over many rounds, each player should receive similar total hearts ... FAIL

FAIL: test_round_score_unbiased (__main__.TestHeartsGame.test_round_score_unbiased)
Over many rounds, each player should receive similar total hearts
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\arpie\AppData\Local\Temp\ipykernel_41640\1016748106.py", line 18, in test_round_score_unbiased
    self.assertAlmostEqual(total / sum(totals), 0.25, delta=0.02)
AssertionError: np.float64(0.3767576923076923) != 0.25 within 0.02 delta (np.float64(0.1267576923076923) difference)

----------------------------------------------------------------------
Ran 1 test in 0.218s

FAILED (failures=1)


<unittest.runner.TextTestResult run=1 errors=0 failures=1>

```
    def round_score(self):
        """Simulate one round of Hearts"""
        scores = []
    	remaining = 13
    	scores = [0] * 4
    	shuffle_list = rng.permutation(4).tolist()
    	for i, player in enumerate(shuffle_list):
    		if i < 3:
    			score = rng.integers(0, remaining, endpoint=True)
    			scores[player] = score  # assign to correct player index
    			remaining -= score
    		else:
    			scores[player] = remaining
    	return scores

        queen = rng.integers(0, 3, endpoint=True)
        scores[queen] += 13
        return scores
```

In [7]:
#del game
import hearts_fixed 
game_f = hearts_fixed.HeartsGame()
i = 1
while i<5000:
    game_f.play("points", 100)
    i+=1
    
print("Standings: ")
for i, p in enumerate(game_f.players):
    print(f"Player {i}: "+ str(game_f.players[i].games_won))

Standings: 
Player 0: 1314
Player 1: 1267
Player 2: 1236
Player 3: 1287


In [8]:
import hearts_fixed 
game =hearts_fixed.HeartsGame()

suite = unittest.TestLoader().loadTestsFromNames(['test_round_score_unbiased'], TestHeartsGame)
unittest.TextTestRunner(verbosity=2).run(suite)

test_round_score_unbiased (__main__.TestHeartsGame.test_round_score_unbiased)
Over many rounds, each player should receive similar total hearts ... ok

----------------------------------------------------------------------
Ran 1 test in 0.362s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

# Other tests 




# Unittest code

We have a lot of functions and there are lots of potential tests we can do.  
We will only highlight a few


## Some things to note:  
- Different class for different types of tests
- TestPlayer, TestHeartsGame, TestIntegration classes
- Each class has own set of tests
- Also we will see a lot of assert statements of different types

![title](assertions.png)

In [9]:
import unittest
import numpy as np

class TestPlayer(unittest.TestCase):
    
    def setUp(self):
        """Create a fresh player before each test"""
        self.player = hearts_fixed.Player(0)
    
    def test_initial_state(self):
        """Player should start with zero score and wins"""
        self.assertEqual(self.player.score, 0)
        self.assertEqual(self.player.games_won, 0)
        self.assertEqual(self.player.player_id, 0)
    
    def test_add_score(self):
        """add_score should accumulate points"""
        self.player.add_score(10)
        self.assertEqual(self.player.score, 10)
        self.player.add_score(5)
        self.assertEqual(self.player.score, 15)
    
class TestHeartsGame(unittest.TestCase):
    
    def setUp(self):
        """Create a fresh game before each test"""
        self.game = hearts_fixed.HeartsGame()
    
    # --- Initialization tests ---
    def test_initial_players(self):
        """Game should start with 4 players"""
        self.assertEqual(len(self.game.players), 4)
    
    def test_initial_rounds(self):
        """Game should start with 0 rounds played"""
        self.assertEqual(self.game.rounds_played, 0)
    
    def test_players_are_player_objects(self):
        """Each player should be a Player instance"""
        for player in self.game.players:
            self.assertIsInstance(player, hearts_fixed.Player)

    def test_round_score_non_negative(self):
        """round_score should never return negative scores"""
        for _ in range(100):
            scores = self.game.round_score()
            for score in scores:
                self.assertGreaterEqual(score, 0)

    # --- play_round tests ---
    def test_play_round_increments_rounds(self):
        """play_round should increment rounds_played"""
        self.game.play_round()
        self.assertEqual(self.game.rounds_played, 1)
    
    # --- get_winners tests ---
    def test_get_winners_returns_lowest_score(self):
        """get_winners should return player(s) with lowest score"""
        self.game.players[0].add_score(10)
        self.game.players[1].add_score(5)   # lowest
        self.game.players[2].add_score(15)
        self.game.players[3].add_score(20)
        winners = self.game.get_winners()
        self.assertEqual(len(winners), 1)
        self.assertEqual(winners[0].player_id, 1)
    
    def test_get_winners_handles_tie(self):
        """get_winners should return all tied players"""
        self.game.players[0].add_score(5)
        self.game.players[1].add_score(5)
        self.game.players[2].add_score(15)
        self.game.players[3].add_score(20)
        winners = self.game.get_winners()
        self.assertEqual(len(winners), 2)

    # --- play tests ---
    
    def test_play_points_exceeds_threshold(self):
        """play with points should stop when a player exceeds threshold"""
        self.game.play("points", 50)
        self.assertTrue(any(p.score >= 50 for p in self.game.players))
   
    def test_play_resets_between_games(self):
        """scores should reset between games but wins should persist"""
        self.game.play("rounds", 5)
        wins_after_first = [p.games_won for p in self.game.players]
        self.game.play("rounds", 5)
        # wins should have grown
        wins_after_second = [p.games_won for p in self.game.players]
        self.assertGreater(sum(wins_after_second), sum(wins_after_first))

    # --- bias test  ---
    def test_round_score_unbiased(self):
        """Over many rounds, each player should receive similar total points"""
        totals = [0] * 4
        for _ in range(5000):
            scores = self.game.round_score()
            for i, score in enumerate(scores):
                totals[i] += score
        total_points = sum(totals)
        for total in totals:
            self.assertAlmostEqual(total / total_points, 0.25, delta=0.05)

class TestIntegration(unittest.TestCase):
    def setUp(self):
        """Create a fresh player before each test"""
        self.player = hearts_fixed.Player(0)
        self.game = hearts_fixed.HeartsGame()
 
    def test_play_round_integration(self):
        """
        Integration test: round_score() output flows correctly 
        through play_round() into Player.add_score()
        """
        # Record scores before
        scores_before = [p.score for p in self.game.players]
        
        # Run play_round -- which calls round_score() and add_score()
        self.game.play_round()
        
        # Record scores after
        scores_after = [p.score for p in self.game.players]
        
        # The difference should sum to 26
        differences = [after - before for after, before 
                       in zip(scores_after, scores_before)]
        self.assertEqual(sum(differences), 26)
        
        # Each player's score should have increased or stayed same
        for diff in differences:
            self.assertGreaterEqual(diff, 0)
        
        # rounds_played should have incremented
        self.assertEqual(self.game.rounds_played, 1)

if __name__ == '__main__':
    unittest.main(argv=['verbose'], verbosity=2,exit=False)

test_get_winners_handles_tie (__main__.TestHeartsGame.test_get_winners_handles_tie)
get_winners should return all tied players ... ok
test_get_winners_returns_lowest_score (__main__.TestHeartsGame.test_get_winners_returns_lowest_score)
get_winners should return player(s) with lowest score ... ok
test_initial_players (__main__.TestHeartsGame.test_initial_players)
Game should start with 4 players ... ok
test_initial_rounds (__main__.TestHeartsGame.test_initial_rounds)
Game should start with 0 rounds played ... ok
test_play_points_exceeds_threshold (__main__.TestHeartsGame.test_play_points_exceeds_threshold)
play with points should stop when a player exceeds threshold ... ok
test_play_resets_between_games (__main__.TestHeartsGame.test_play_resets_between_games)
scores should reset between games but wins should persist ... ok
test_play_round_increments_rounds (__main__.TestHeartsGame.test_play_round_increments_rounds)
play_round should increment rounds_played ... ok
test_players_are_player

In [10]:
# Running subset of tests 
suite = unittest.TestLoader().loadTestsFromNames(['test_play_resets_between_games', 'test_round_score_unbiased'], TestHeartsGame)
unittest.TextTestRunner(verbosity=2).run(suite)

test_play_resets_between_games (__main__.TestHeartsGame.test_play_resets_between_games)
scores should reset between games but wins should persist ... ok
test_round_score_unbiased (__main__.TestHeartsGame.test_round_score_unbiased)
Over many rounds, each player should receive similar total points ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.193s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

In [11]:
# Running all tests
unittest.main(argv=['verbose'], verbosity=2,exit=False)

test_get_winners_handles_tie (__main__.TestHeartsGame.test_get_winners_handles_tie)
get_winners should return all tied players ... ok
test_get_winners_returns_lowest_score (__main__.TestHeartsGame.test_get_winners_returns_lowest_score)
get_winners should return player(s) with lowest score ... ok
test_initial_players (__main__.TestHeartsGame.test_initial_players)
Game should start with 4 players ... ok
test_initial_rounds (__main__.TestHeartsGame.test_initial_rounds)
Game should start with 0 rounds played ... ok
test_play_points_exceeds_threshold (__main__.TestHeartsGame.test_play_points_exceeds_threshold)
play with points should stop when a player exceeds threshold ... ok
test_play_resets_between_games (__main__.TestHeartsGame.test_play_resets_between_games)
scores should reset between games but wins should persist ... ok
test_play_round_increments_rounds (__main__.TestHeartsGame.test_play_round_increments_rounds)
play_round should increment rounds_played ... ok
test_players_are_player